# Predicting F1 Pit Stops

In [1]:
import os
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import ExtraTreesClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

## Data Read

In [2]:
# Kaggle Notebook
# df_train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e5/train.csv')
# df_test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e5/test.csv')

# local 
data_path = "/mnt/e/Kaggle_Competitions/Predicting_F1_Pit_Stops"
train_dp = os.path.join(data_path, "train.csv")
test_dp = os.path.join(data_path, "test.csv")

df_train = pd.read_csv(train_dp)
df_test = pd.read_csv(test_dp)

In [3]:
df_train.head()

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0


In [4]:
df_train.describe()

,id,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
count,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000
mean,219569.500000,2023.523544,0.136118,23.105909,1.789113,14.158231,9.630339,90.948735,-3.770040,-25.721759,0.337661,0.101542,0.198982
std,126768.942943,1.024930,0.342915,16.958261,0.950194,9.801338,5.278770,19.772769,43.945759,54.766573,0.253277,4.006765,0.399235
min,0.000000,2022.000000,0.000000,1.000000,1.000000,1.000000,1.000000,67.694000,-2403.895000,-274.564000,0.012821,-18.000000,0.000000
25%,109784.750000,2023.000000,0.000000,9.000000,1.000000,6.000000,5.000000,82.621000,-8.884000,-46.566250,0.129870,-1.000000,0.000000
50%,219569.500000,2024.000000,0.000000,19.000000,2.000000,12.000000,10.000000,90.521000,-0.295000,-20.994000,0.269231,0.000000,0.000000
75%,329354.250000,2024.000000,0.000000,36.000000,2.000000,20.000000,14.000000,98.471000,0.115000,-6.199000,0.513158,2.000000,0.000000
max,439139.000000,2025.000000,1.000000,78.000000,8.000000,77.000000,20.000000,2507.607000,2423.932000,2412.026000,1.000000,18.000000,1.000000


In [5]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 439140 entries, 0 to 439139
Data columns (total 16 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   id                      439140 non-null  int64  
 1   Driver                  439140 non-null  object 
 2   Compound                439140 non-null  object 
 3   Race                    439140 non-null  object 
 4   Year                    439140 non-null  int64  
 5   PitStop                 439140 non-null  int64  
 6   LapNumber               439140 non-null  int64  
 7   Stint                   439140 non-null  int64  
 8   TyreLife                439140 non-null  float64
 9   Position                439140 non-null  int64  
 10  LapTime (s)             439140 non-null  float64
 11  LapTime_Delta           439140 non-null  float64
 12  Cumulative_Degradation  439140 non-null  float64
 13  RaceProgress            439140 non-null  float64
 14  Position_Change     

## Data Analysis

In [6]:
# df_train['PitStop'].hist(bins=50)
# plt.show()

In [7]:
numeric_cols = df_train.select_dtypes(include=['int64', 'float64']).columns

corr = df_train[numeric_cols].corr()

corr['PitStop'].sort_values(ascending=False)

PitStop                   1.000000
Stint                     0.143869
LapNumber                 0.111431
Position                  0.093536
Year                      0.090767
RaceProgress              0.059177
PitNextLap                0.048567
id                       -0.000214
LapTime (s)              -0.000437
LapTime_Delta            -0.008458
Cumulative_Degradation   -0.088052
Position_Change          -0.107659
TyreLife                 -0.134678
Name: PitStop, dtype: float64

In [8]:
# for col in numeric_cols[:10]:
#     plt.figure(figsize=(6,3))
#     df_train[col].hist(alpha=0.5, label='train')
#     df_test[col].hist(alpha=0.5, label='test')
#     plt.legend()
#     plt.title(col)
#     plt.show()

## Dataset Preprocessing

In [9]:
TARGET = 'PitNextLap'

dset_train = df_train.copy()
dset_test = df_test.copy()

X = dset_train.drop(columns=[TARGET, 'id'])
y = dset_train[TARGET]

X_test = dset_test.drop(columns=['id'])

In [10]:
cat_cols = X.select_dtypes(include='object').columns

for col in cat_cols:

    le = LabelEncoder()

    full_data = pd.concat([
        X[col],
        X_test[col]
    ]).astype(str)

    le.fit(full_data)

    X[col] = le.transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

## Model Train and Test

In [11]:
# training and test setup
n_splits = 3
skf = StratifiedKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=42
)

oof = np.zeros(len(X))
rf_preds = np.zeros(len(X_test))

### Random Forest Model

In [12]:
rf_preds = np.zeros(len(X_test))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):

    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_valid = X.iloc[valid_idx]
    y_valid = y.iloc[valid_idx]

    # Random Forest
    rf_model = RandomForestClassifier(
        n_estimators=500,
        max_depth=30,
        min_samples_leaf=10,
        n_jobs=-1,
        random_state=42,
        class_weight='balanced'
    )

    # train
    rf_model.fit(X_train, y_train)

    # valid
    rf_valid_pred = rf_model.predict_proba(X_valid)[:,1]
    oof[valid_idx] = rf_valid_pred
    score = roc_auc_score(y_valid, rf_valid_pred)
    print(f'Fold {fold + 1}: {score:.5f}')

    # test
    rf_preds += rf_model.predict_proba(X_test)[:,1] / n_splits

Fold 1: 0.94257
Fold 2: 0.94178
Fold 3: 0.94209


### ExtraTrees Model

In [16]:
et_preds = np.zeros(len(X_test))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):

    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_valid = X.iloc[valid_idx]
    y_valid = y.iloc[valid_idx]

    et_model = ExtraTreesClassifier(
        n_estimators=350,
        max_depth=25,
        min_samples_leaf=10,
        max_features=0.3,
        bootstrap=True,
        n_jobs=-1,
        random_state=42,
        class_weight='balanced'
    )

    et_model.fit(X_train, y_train)

    et_valid_pred = et_model.predict_proba(X_valid)[:,1]
    oof[valid_idx] = et_valid_pred
    score = roc_auc_score(y_valid, et_valid_pred)
    print(f'Fold {fold + 1}: {score:.5f}')

    et_preds += et_model.predict_proba(X_test)[:,1] / n_splits

Fold 1: 0.93547
Fold 2: 0.93445
Fold 3: 0.93451


### CatBoost Model

In [14]:
cb_preds = np.zeros(len(X_test))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):

    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_valid = X.iloc[valid_idx]
    y_valid = y.iloc[valid_idx]

    cb_model = CatBoostClassifier(
        iterations=3000,
        learning_rate=0.1,
        depth=10,
        loss_function='Logloss',
        eval_metric='AUC',
        random_seed=42,
        early_stopping_rounds=100,
        verbose=500,
        allow_writing_files=False
    )


    cb_model.fit(X_train, y_train)

    cb_valid_pred = cb_model.predict_proba(X_valid)[:,1]
    oof[valid_idx] = cb_valid_pred
    score = roc_auc_score(y_valid, cb_valid_pred)
    print(f'Fold {fold + 1}: {score:.5f}')

    cb_preds += cb_model.predict_proba(X_test)[:,1] / n_splits

0:	total: 107ms	remaining: 5m 19s
500:	total: 24s	remaining: 1m 59s
1000:	total: 47.1s	remaining: 1m 33s
1500:	total: 1m 10s	remaining: 1m 10s
2000:	total: 1m 34s	remaining: 47.1s
2500:	total: 1m 58s	remaining: 23.6s
2999:	total: 2m 21s	remaining: 0us
Fold 1: 0.94391
0:	total: 44ms	remaining: 2m 11s
500:	total: 23.4s	remaining: 1m 56s
1000:	total: 48.8s	remaining: 1m 37s
1500:	total: 1m 13s	remaining: 1m 13s
2000:	total: 1m 38s	remaining: 49.2s
2500:	total: 2m 2s	remaining: 24.5s
2999:	total: 2m 25s	remaining: 0us
Fold 2: 0.94278
0:	total: 48.2ms	remaining: 2m 24s
500:	total: 23.6s	remaining: 1m 57s
1000:	total: 47.4s	remaining: 1m 34s
1500:	total: 1m 10s	remaining: 1m 10s
2000:	total: 1m 34s	remaining: 47.3s
2500:	total: 1m 57s	remaining: 23.4s
2999:	total: 2m 19s	remaining: 0us
Fold 3: 0.94314


## Test Result

In [18]:
final_preds = 0.4 * rf_preds + 0 * et_preds + 0.6 * cb_preds
# final_preds = cb_preds

submission = pd.DataFrame({
    'id': df_test['id'],
    'PitNextLap': final_preds
})

preds_save_path = os.path.join(data_path, 'submission.csv')
submission.to_csv(preds_save_path, index=False)